# Sharded image training

The corpus is generated here, then one pipeline packs it and starts distributed training.

1. Generate 20,000 labeled 256x256 JPEGs on the shared volume.
2. The `pack_shards` pipeline step writes WebDataset tar shards.
3. The `train` step submits one TrainJob, ResNet18 from scratch, two L40S GPUs.
4. After the next cell launches the run, open **Develop and train → Pipelines → Runs**.

In [ ]:
from generate_images import CLASSES, count_images, generate

RAW = "/opt/app-root/src/data/raw"
generate(20000, RAW)
print("classes:", ", ".join(CLASSES))
print("image_count:", count_images(RAW))

In [ ]:
from generate_images import sample_paths
from PIL import Image
import matplotlib.pyplot as plt

paths = sample_paths(RAW)
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, path in zip(axes.ravel(), paths):
    ax.imshow(Image.open(path))
    ax.set_title(path.parent.name)
    ax.axis("off")
fig.tight_layout()
plt.show()

## Prep

`pack_shards.py` reads `/data/raw/<class>/*.jpg` and writes `/data/shards/shard-*.tar`.
Each sample is `000123.jpg` plus `000123.cls`. The step emits a small manifest
(path, image count, shard count, bytes). The image bytes stay on the shared volume.

## Train

`submit_trainjob.py` creates one TrainJob on runtime `torch-distributed-cuda130-torch211-py312`.
Two nodes, one GPU each, run `train.py`: ResNet18 from scratch, DDP, WebDataset
`split_by_node` / `split_by_worker`, `num_workers=2`, batch 128, one epoch.
The GPU pods are the TrainJob, not extra boxes in the pipeline graph.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--index-url", "https://pypi.org/simple", "kfp==2.17.0"])
from pipeline import submit

submit()